In [1]:
import pandas as pd
import numpy as np
import json
import re
from num2words import num2words
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

/root/master_thesis/thesis_multi_speaker_asr/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
metadata = pd.read_csv('/root/master_thesis/thesis_multi_speaker_asr/data/CORAAL/metadata.csv')

In [3]:
metadata = pd.read_csv('/root/master_thesis/thesis_multi_speaker_asr/data/commonvoice-en/cv-corpus-25.0-2026-03-09/en/cv_metadata.csv')

In [21]:
def fetch_results(path):    
    ids = []
    text = []
    start_time = []
    end_time = []
    rtf = []

    df = pd.DataFrame()

    with open(path, 'r') as file:
        lines = file.read().splitlines()

        for line in lines:
            json_line = json.loads(line)
            print(json_line)
            ids.append(list(json_line)[0])
            for item in json_line.values():
                segment = item['segments']
                print(segment)
                if len(segment) == 0:
                    text.append('NaN')
                    start_time.append(np.nan)
                    end_time.append(np.nan)
                else:
                    segment = segment[0]
                    text.append(segment['text'])
                    start_time.append(segment['start'])
                    end_time.append(segment['end'])
                rtf.append(item['rtf'])


    df['id'] = ids
    df['text'] = text
    df['start'] = start_time
    df['end'] = end_time
    df['rtf'] = rtf

    print(len(df['text']))
    print(df['text'])

    return df

In [5]:
def calculate_wer(reference, hypothesis):
	ref_words = reference.split()
	hyp_words = hypothesis.split()
	# Counting the number of substitutions, deletions, and insertions
	substitutions = sum(1 for ref, hyp in zip(ref_words, hyp_words) if ref != hyp)
	deletions = len(ref_words) - len(hyp_words)
	insertions = len(hyp_words) - len(ref_words)
	# Total number of words in the reference text
	total_words = len(ref_words)
	# Calculating the Word Error Rate (WER)
	wer = (substitutions + deletions + insertions) / total_words
	return wer

In [6]:
def clean_text(sentence):
    sentence = str.lower(sentence)
    sentence = re.sub(r'-(?!\d)', '', sentence)             # Remove - that are not followed by a number
    sentence = re.sub(r'(?<!\d)\.|\.?(?!\d)', '', sentence) # Remove . that are not enclosed by two numbers
    sentence = re.sub(r'[^\w\s.-]', '', sentence)           # Remove all punctuation except for the - and .
    
    sentence_copy = str(sentence)

    for s in sentence.split():
        try: 
            num = float(s)
            word_rep = str(num2words(number=num))
            sentence_copy = sentence_copy.replace(s, word_rep)
        except ValueError as e:
            continue

    return sentence_copy    

In [7]:
def clean(df):
    # Clean hypothesis and references before WER calculation:
    hypothesis_clean = []
    reference_clean = []
    for _, row in df.iterrows():
        hypothesis = row['hypothesis']
        reference = row['reference']
        hyp = clean_text(hypothesis)
        ref = clean_text(reference)
        hypothesis_clean.append(hyp)
        reference_clean.append(ref)

    df['hypothesis'] = hypothesis_clean
    df['reference'] = reference_clean
    
    return df

In [8]:
def compute_all_wer(df):
    df = clean(df)

    # Compute WER for all output samples:
    wer_vals = []
    for _, row in df.iterrows():
        wer = calculate_wer(hypothesis=row['hypothesis'],
                            reference=row['reference'])
        wer_vals.append(wer)

    df['wer'] = wer_vals
    # Calculate average WER accross the whole test split:
    return df

In [ ]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cuda')

In [9]:
def compute_all_semdist(df):
    
    df = clean(df)

    emb_hyp = embedding_model.encode(df['hypothesis'].tolist(), device='cuda', convert_to_numpy=True, batch_size=128)
    emb_ref = embedding_model.encode(df['reference'].tolist(), device='cuda', convert_to_numpy=True, batch_size=128)
    
    print(emb_hyp.shape)
    print(emb_ref.shape)

    semdist = embedding_model.similarity(emb_hyp, emb_ref).flatten().tolist()
    print(semdist)

    return df

In [ ]:
df_tiny = fetch_results('/root/master_thesis/thesis_multi_speaker_asr/src/results/hpc_results/tiny_int8_cpu_threads_4.jsonl')
df_base = fetch_results('/root/master_thesis/thesis_multi_speaker_asr/src/results/hpc_results/base_int8_cpu_threads_4.jsonl')
df_small = fetch_results('/root/master_thesis/thesis_multi_speaker_asr/src/results/hpc_results/small_int8_cpu_threads_4.jsonl')
df_medium = fetch_results('/root/master_thesis/thesis_multi_speaker_asr/src/results/hpc_results/medium_int8_cpu_threads_4.jsonl')

print(f'tiny: {len(df_tiny)}, base: {len(df_base)}, small: {len(df_small)}, medium: {len(df_medium)}')

In [23]:
#metadata = pd.read_csv('/root/master_thesis/thesis_multi_speaker_asr/data/commonvoice-en/cv-corpus-25.0-2026-03-09/en/cv_metadata.csv')

base_model_results = '/root/master_thesis/thesis_multi_speaker_asr/src/results/hpc_results/base_int8_cpu_threads_4.jsonl'
tiny_model_results = '/root/master_thesis/thesis_multi_speaker_asr/src/results/hpc_results/tiny_int8_cpu_threads_4.jsonl'
small_model_results = '/root/master_thesis/thesis_multi_speaker_asr/src/results/hpc_results/small_int8_cpu_threads_4.jsonl'
medium_model_results = '/root/master_thesis/thesis_multi_speaker_asr/src/results/hpc_results/medium_int8_cpu_threads_4.jsonl'

tiny_coraal_results = '/root/master_thesis/thesis_multi_speaker_asr/src/results/tiny_cpu_int8_threads_4.jsonl'


res_list = [
    ('tiny', tiny_model_results),
    ('base', base_model_results),
    ('small', small_model_results),
    ('medium', medium_model_results)
]

res_list = [('tiny_coraal', tiny_coraal_results)]

for name, path in res_list:
    df = fetch_results(path)
    #df = df.merge(metadata, on='id')
    #df = df.rename(columns={'text': 'hypothesis', 'sentence': 'reference'})



    #df = compute_all_wer(df)
    #avg_wer_total = df['wer'].mean()
    #print(f'Model: {name}, Avg. WER: {avg_wer_total}')

    #avg_rtf = df['rtf'].mean()
    #print(f'Model: {name}, Avg. RTF: {avg_rtf}')

    #df = compute_all_semdist(df)
    #avg_semdist = df['semdist'].mean()
    #print(f'Model: {name}, Avg. Semdist: {avg_semdist}')


{'ccfe847f-dff7-41b1-96da-db13f229df73': {'segments': [{'start': 0.5, 'end': 33.82, 'text': " Okay. How's it going? Of course, my name is... I'm a goddess gonna be interviewer. You and today, can I start by your name? What's your name? My name is... Okay, cool, cool. Um, the name lie down, I'm assuming a female, but for the record, are you female? I'm a female. Okay, your ethnicity? I am too, too, and more racist, too, more racist."}, {'start': 33.82, 'end': 63.86, 'text': " Which is... I'm black. And then I have a little bit of Spaniard. Okay, cool, cool. No mind me, I want you to kind of open up and give me some long, windy type answers. Okay. What year were you born? 1984. Okay. Of April. 1984 of April. What sign would that make you? I'm an Aries."}, {'start': 63.86, 'end': 93.39, 'text': " Very feisty. Okay. That's cool. That's what's up. We can talk. Loyal. We can possibly talk about that in a little bit. Let me see what else. Let me just go down the checklist right quick. What's 

In [14]:
df.head(10)

,id,hypothesis,start,end,rtf,CORAAL.Sub,Version.Created,Version.Modified,CORAAL.Spkr,CORAAL.File,...,Sampling.Rate,Source.Device,Dig.Sampling.Rate,Dig.Bit.Rate,Dig.Channels,CORAAL.Length.of.Transcript,CORAAL.Word.Count,Is.Misc.Tier,Notes,path
0,ccfe847f-dff7-41b1-96da-db13f229df73,"Okay. How's it going? Of course, my name is.....",0.50,33.82,0.022592,ATL,v.2020.05,NaN,ATL_se0_ag2_f_01,ATL_se0_ag2_f_01_1,...,44.1 kHz,NaN,44.1 khz,16 bit,Mono,2339.8,6881,NaN,NaN,ATL_se0_ag2_f_01_1.wav
1,d3121dd6-4ab1-4fbb-b680-163196558ea6,"Okay, this is another interview from Oregon S...",0.14,40.93,0.025754,ATL,v.2020.05,NaN,ATL_se0_ag2_m_01,ATL_se0_ag2_m_01_1,...,44.1 kHz,NaN,44.1 khz,16 bit,Mono,2440.6,7166,NaN,NaN,ATL_se0_ag2_m_01_1.wav
2,a9348157-e506-4da8-94b6-615ba3096a14,"Hey, what's going on? I'm here with by the wa...",0.37,38.19,0.016471,ATL,v.2020.05,NaN,ATL_se0_ag2_f_02,ATL_se0_ag2_f_02_1,...,44.1 kHz,NaN,44.1 khz,16 bit,Mono,2497.4,5252,yes,NaN,ATL_se0_ag2_f_02_1.wav
3,6be8e426-4757-4672-ae02-76ab69cc7caf,"Yo, um, this is an interview and what's the n...",0.91,46.62,0.019505,ATL,v.2020.05,NaN,ATL_se0_ag1_m_01,ATL_se0_ag1_m_01_1,...,44.1 kHz,NaN,44.1 khz,16 bit,Mono,2755.3,7223,NaN,NaN,ATL_se0_ag1_m_01_1.wav
4,3fe519d3-5a20-4b95-8459-b72fef046216,It's about don't say him to his daddy. You ju...,0.08,36.10,0.018437,ATL,v.2020.05,NaN,ATL_se0_ag1_f_01,ATL_se0_ag1_f_01_1,...,44.1 kHz,NaN,44.1 khz,16 bit,Mono,1862.0,4586,NaN,NaN,ATL_se0_ag1_f_01_1.wav
5,2f339e9d-4ea0-4538-904d-5d58653ed93b,"Yeah, I'm here with... Who am I with? Motherf...",0.00,36.78,0.023499,ATL,v.2020.05,NaN,ATL_se0_ag1_m_02,ATL_se0_ag1_m_02_1,...,44.1 kHz,NaN,44.1 khz,16 bit,Mono,2434.5,6955,NaN,NaN,ATL_se0_ag1_m_02_1.wav
6,8000f87a-0be9-41a6-bc82-e98c2e8b09ab,My government name is from originally from Co...,0.66,30.58,0.025632,ATL,v.2020.05,NaN,ATL_se0_ag1_m_03,ATL_se0_ag1_m_03_1,...,44.1 kHz,NaN,44.1 khz,16 bit,Mono,2759.5,10198,NaN,NaN,ATL_se0_ag1_m_03_1.wav
7,2744530f-6863-4483-bba3-7d371acc2cb8,Can you tell me your name? I was named by the...,0.53,35.20,0.024573,ATL,v.2020.05,NaN,ATL_se0_ag2_m_02,ATL_se0_ag2_m_02_1,...,44.1 kHz,NaN,44.1 khz,16 bit,Mono,2545.1,8251,NaN,NaN,ATL_se0_ag2_m_02_1.wav
8,7aaa76ab-66a5-41e4-8189-28a5597dc796,"Okay, um, my name is and today I will be inte...",0.94,33.39,0.024252,ATL,v.2020.05,NaN,ATL_se0_ag1_f_02,ATL_se0_ag1_f_02_1,...,44.1 kHz,NaN,44.1 khz,16 bit,Mono,2210.2,6687,NaN,NaN,ATL_se0_ag1_f_02_1.wav
9,e58c7a2d-7605-48f9-9747-3567ab913498,"Can I get your full name, please? Okay, and w...",0.27,31.06,0.024684,ATL,v.2020.05,NaN,ATL_se0_ag1_f_03,ATL_se0_ag1_f_03_1,...,44.1 kHz,NaN,44.1 khz,16 bit,Mono,1808.2,4946,yes,NaN,ATL_se0_ag1_f_03_1.wav


In [ ]:
df = fetch_results(medium_model_results)
df = df.merge(metadata, on='id')
df = df.rename(columns={'text': 'hypothesis', 'sentence': 'reference'})



hyp = df['hypothesis'].tolist()
ref = df['reference'].tolist()

simi = []
for (h, r) in tqdm(zip(hyp, ref)):
    
    emb_hyp = embedding_model.encode(h, device='cuda', convert_to_tensor=True)
    emb_ref = embedding_model.encode(r, device='cuda', convert_to_tensor=True)

    sim = 1 - util.cos_sim(emb_hyp, emb_ref)
    sim = float(sim.flatten().cpu().numpy()[0])
    sim = round(sim, 3)

    #print(f'Hyp: {h}, Ref: {r}, Dist: {sim}')
    simi.append(sim)


df['semdist'] = simi
avg_semdist = df['semdist'].mean()
print(avg_semdist)
df.head(10)
    

In [ ]:
df['semdist'] = simi
avg_semdist = df['semdist'].mean()
print(avg_semdist)
df.head(10)


In [ ]:
# Calculate total duration for the test split:
seconds = 49334
hours = seconds // 3600
hours_remainder = seconds % 3600
minutes = hours_remainder // 60
seconds_remainder = hours_remainder % 60
time = '%02d:%02d:%02d'%(hours, minutes, seconds_remainder)
print(time)

print(f"Total CPU time in hours: {seconds / 3600:.2f}")